# Step 6: one attention head

This notebook goes with step 6 of `vak`. It runs attention at its smallest size: three tokens, each with vectors of
two numbers, small enough to check by hand. `uv run vak attention` runs the same code.

Before this: step 5 writes with a model that reads only the last token. After this: step 7 runs many heads at once,
and adds position.

| Section | Idea | Used in |
|---|---|---|
| 1 | Attention, step by step, with the causal mask | `vak attention` |
| 2 | The bank row by hand | `vak attention` |
| 3 | New neighbours give a new output | `vak attention` |

## 1. Attention, step by step

**Attention** lets each token read the tokens before it. Each token has a **query**, what it looks for; a **key**,
what it offers; and a **value**, what it passes on. A token's **score** for another token is its query times that
token's key. Softmax turns each row of scores into **weights** that add up to 1, and the output is the weighted mix
of the values. Here are three tokens, "The river bank", with vectors of two numbers, read as "water" and "money".

In [1]:
import torch
from medha.model import attention

tokens = ["The", "river", "bank"]
q = torch.tensor([[0.0, 0.0], [0.5, 0.5], [1.5, 1.5]])
k = torch.tensor([[0.0, 0.0], [2.0, 0.0], [0.5, 0.5]])
v = torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.5, 0.5]])
steps = attention(q, k, v)
for token, scores, weights in zip(tokens, steps.masked.tolist(), steps.weights.tolist()):
    print(f"{token:6} masked scores {[round(score, 3) for score in scores]}   weights {[round(w, 3) for w in weights]}")
print("bank's output:", [round(number, 3) for number in steps.out[2].tolist()])

The    masked scores [0.0, -inf, -inf]   weights [1.0, 0.0, 0.0]
river  masked scores [0.0, 0.707, -inf]   weights [0.33, 0.67, 0.0]
bank   masked scores [0.0, 2.121, 1.061]   weights [0.082, 0.682, 0.236]
bank's output: [0.8, 0.118]


**What you see**

Each row is one token reading. The **causal mask** puts minus infinity wherever a token would read a later token,
so "The" has scores `[0.0, -inf, -inf]` and reads only itself, with weight 1. Softmax turns minus infinity into a
weight of exactly 0. "bank" reads all three, with weights `[0.082, 0.682, 0.236]`. Its output is mostly the value
of "river": `[0.8, 0.118]`, high on "water".

Used in: `uv run vak attention` prints each step as a 3 × 3 table: scores, scaled, masked, weights, then the output.

## 2. The bank row by hand

Now the bank row with nothing but `math`. Its query (1.5, 1.5) times each key gives the scores. Divide each by the
square root of the head size, 2. Then softmax: raise e to each number, and divide each by their sum.

In [2]:
import math

scores = [1.5 * 0 + 1.5 * 0, 1.5 * 2 + 1.5 * 0, 1.5 * 0.5 + 1.5 * 0.5]  # times the keys of The, river and bank
scaled = []
for score in scores:
    scaled.append(score / math.sqrt(2))
exps = []
for number in scaled:
    exps.append(math.exp(number))
weights = []
for number in exps:
    weights.append(number / sum(exps))
out = [weights[1] * 1.0 + weights[2] * 0.5, weights[2] * 0.5]  # The's value is (0, 0), so it adds nothing
print("scores ", scores)
print("scaled ", [round(number, 3) for number in scaled])
print("e^     ", [round(number, 3) for number in exps], "add up to", round(sum(exps), 3))
print("weights", [round(number, 3) for number in weights])
print("output ", [round(number, 3) for number in out])

scores  [0.0, 3.0, 1.5]
scaled  [0.0, 2.121, 1.061]
e^      [1.0, 8.342, 2.888] add up to 12.23
weights [0.082, 0.682, 0.236]
output  [0.8, 0.118]


**What you see**

The scores are `[0.0, 3.0, 1.5]`: the key of "river" matches the query best. Dividing by 1.414 gives
`[0.0, 2.121, 1.061]`. Raised to e, they are `[1.0, 8.342, 2.888]`, which `add up to 12.23`. Each divided by that
sum gives the weights `[0.082, 0.682, 0.236]`, the same as the bank row in section 1. The output `[0.8, 0.118]`
matches too.

Used in: the same numbers are the bank row of the tables that `uv run vak attention` prints.

## 3. New neighbours give a new output

The embedding gives a token the same vector in every sentence. Attention changes that. Put "loan" where "river"
was, with the key (0, 2) and the value (0, 1), and run the same query of "bank" again.

In [3]:
k_loan = k.clone()
v_loan = v.clone()
k_loan[1] = torch.tensor([0.0, 2.0])
v_loan[1] = torch.tensor([0.0, 1.0])
loan = attention(q, k_loan, v_loan)
print("bank's weights:", [round(number, 3) for number in loan.weights[2].tolist()])
print("bank's output: ", [round(number, 3) for number in loan.out[2].tolist()])

bank's weights: [0.082, 0.682, 0.236]
bank's output:  [0.118, 0.8]


**What you see**

The query of "bank" matches the key of "loan" as well as it matched "river", so the weights stay
`[0.082, 0.682, 0.236]`. But the values differ, so the output turns to `[0.118, 0.8]`, high on "money". Same token,
new neighbours, new vector.

Used in: the last two lines of `uv run vak attention`. The GPT in step 8 runs the same `attention` in every block.

## Recap

| Idea | Remember | Used next |
|---|---|---|
| Attention | weights = softmax(q · k / √(head size)), row by row; output = weights × values | step 7 runs it for many heads at once |
| Causal mask | minus infinity above the diagonal, so no token reads a later one | every block of the GPT, step 8 |
| By hand | bank: scores 0, 3, 1.5; weights 0.082, 0.682, 0.236; output (0.8, 0.118) | a check for any change to `attention` |
| Context | the same token gets a new output when its neighbours change | why a GPT reads more than one token |